# Ingestão de Dados de Ações

Este notebook coleta dados históricos de ações brasileiras e americanas dos últimos 5 anos usando a biblioteca `yfinance`, consolida em um único DataFrame e salva em formato Parquet e tabela Delta.

**Tickers incluídos:**

* **Brasileiras (B3):** PETR4.SA, VALE3.SA, ITUB4.SA, BBAS3.SA, WEGE3.SA, B3SA3.SA
* **Americanas (NASDAQ/NYSE):** AAPL, MSFT, NVDA, GOOGL, AMZN, META

In [0]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# Pasta de saída
PASTA_DATA = Path("data")
PASTA_DATA.mkdir(parents=True, exist_ok=True)

# Ações brasileiras e americanas
tickers = [
    "PETR4.SA", "VALE3.SA", "ITUB4.SA",
    "BBAS3.SA", "WEGE3.SA", "B3SA3.SA",
    "AAPL", "MSFT", "NVDA", "GOOGL",
    "AMZN", "META"
]

# Últimos 5 anos
df = yf.download(
    tickers,
    period="5y",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    progress=False
)

# Transforma os dados em uma tabela única
frames = []

for ticker in tickers:
    try:
        dados = df[ticker].copy()
        dados = dados.dropna(how="all")

        if dados.empty:
            continue

        dados = dados.reset_index()
        dados["ticker"] = ticker
        frames.append(dados)

    except KeyError:
        print(f"Sem dados disponíveis: {ticker}")

# Consolida e salva em Parquet
if frames:
    resultado = pd.concat(frames, ignore_index=True)

    resultado.columns = [
        str(col).lower().replace(" ", "_")
        for col in resultado.columns
    ]

    # Caminho do arquivo de saída
    arquivo_saida = PASTA_DATA / "acoes_ultimos_5_anos.parquet"

    resultado.to_parquet(
        arquivo_saida,
        index=False
    )

    print(f"Arquivo criado: {arquivo_saida}")
    print(f"Total de registros: {len(resultado)}")
else:
    print("Nenhum dado foi obtido.")

# Visualização
Carrega o arquivo Parquet gerado e exibe as primeiras linhas para conferir os dados.

In [0]:
df = pd.read_parquet("data/acoes_ultimos_5_anos.parquet")
display(df.head(10))

## Persistência em Tabela Delta

Converte o DataFrame pandas para Spark e salva como tabela Delta no Unity Catalog (`main.default.acoes_ultimos_5_anos`).

In [0]:
# Converte o DataFrame pandas para Spark e salva em tabela Delta
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.acoes_ultimos_5_anos")

# Verificação
print(f"Registros salvos na tabela Delta: {spark_df.count()}")
print("Tabela: main.default.acoes_ultimos_5_anos")